# End-to-End LLM / VLM / RAG / AgentOps / LLMOps — Master Notebook

> **The single, canonical, deduplicated notebook for the GenAI Engineering Hub.**
> Maps 1:1 to the 12 sections in `docs/` and ships with runnable examples in `examples/`.

**Version:** V1.4 Master · **Last updated:** 2026-06-22

---

## 📑 Quick Navigation

| # | Section | What You'll Learn |
|---:|---|---|
| 1 | LLM Architecture | Transformer → RoPE → Attention → MoE → Beyond-Transformer |
| 2 | Attention & Serving Kernels | FlashAttention, PagedAttention, KV-cache, inference optimization |
| 3 | VLM & Multimodal | CLIP → BLIP-2 → LLaVA → Qwen-VL; multimodal embeddings |
| 4 | Coding Models | Why coding models are different; the coding landscape |
| 5 | Fine-Tuning & PEFT | LoRA, QLoRA, DoRA, full FT, capacity planning |
| 6 | Alignment & RLHF | PPO → DPO → ORPO → SimPO → GRPO |
| 7 | RAGOps | Basic RAG → Hybrid → Reranker → Multimodal → GraphRAG |
| 8 | AgentOps | ReAct, Planner-Executor, LangGraph, memory, multi-agent |
| 9 | LLMOps & EvalOps | Prompt registry, evaluation, observability, version control |
| 10 | AWS Production | Bedrock, EKS+KServe, vLLM, SageMaker, Ray |
| 11 | Security & Guardrails | Prompt injection, RAG poisoning, memory safety, PII |
| 12 | End-to-End Blueprint | The complete production system map |

---

## 🧭 Core Principle

```text
Architecture tells which model to choose.
Fine-tuning tells how to adapt it.
RAG connects private/external knowledge.
Agents connect tools, workflows and memory.
Serving runs the system at scale.
LLMOps evaluates, monitors and rolls back.
Security prevents leakage, poisoning and abuse.
```

## 📖 How To Use This Notebook

1. **Linear reader?** Read top-to-bottom — each section is self-contained.
2. **Reference?** Use the Quick Navigation table above to jump.
3. **Looking for code?** Every section links to its runnable example in `examples/`.
4. **Want to contribute?** See `CONTRIBUTING.md` and `references/papers.md`.

**Maturity convention used throughout:**

```text
main     = used in a released model / major report / widely deployed framework
watchlist = emerging / niche / unconfirmed adoption — included for completeness, not endorsed
```


# Section 1 — LLM Architecture

> **What this section is for:** Understand what's happening inside modern LLMs — positional encoding, attention variants, FFN/MoE, and beyond-transformer alternatives. You need this before you can read a 2025–2026 model report.


## 1.0 — Reading Guide

This is not a benchmark leaderboard. It's an **architecture reference map** for reading modern LLM reports.

Roadmap flow:

```text
Base Transformer
→ Positional Encoding upgrades
→ Attention / KV-cache upgrades
→ FFN / MoE upgrades
→ Multimodal architecture
→ Beyond-Transformer alternatives
→ Model-wise technical reports
```

For each paper / report, capture:

```text
1. What problem does it solve?
2. Which block changed?
3. What is the new mechanism?
4. Which model uses it?
5. What trade-off does it introduce?
6. What one-line explanation can I teach?
```


## 1.1 — Base Transformer Refresher

### Core Block

```text
Input Tokens
   ↓
Token Embedding
   ↓
Position Encoding / RoPE
   ↓
Attention Block
   ↓
FFN / MLP Block
   ↓
Residual + Norm
   ↓
Next Layer
```

### Component Roles

| Component | Role |
|---|---|
| Token Embedding | Converts token IDs into vectors |
| Positional Encoding | Tells model where tokens are |
| Attention | Mixes information across tokens |
| FFN / MLP | Transforms each token internally |
| MoE | Replaces one FFN with many expert FFNs |
| KV Cache | Stores keys/values for fast autoregressive decoding |
| Router | Selects which experts are used per token |

> 📌 **Cross-ref:** [`docs/01-architecture/README.md`](docs/01-architecture/README.md)


## 1.2 — Positional Encoding

**Why:** Transformers are permutation-invariant — without position info, "dog bites man" and "man bites dog" look the same.

### Evolution

```text
Sinusoidal PE → Learned Absolute PE → Relative Position Bias → ALiBi → RoPE
→ RoPE base/theta scaling → NTK-aware RoPE → Dynamic RoPE
→ Position Interpolation → YaRN → LongRoPE → LongRoPE2
→ MRoPE → TMRoPE → Interleaved-MRoPE → MHRoPE / MRoPE-I
→ NoPE → iRoPE → P-RoPE / Periodic RoPE → HoPE variants → LazyAttention
```

### Cheat Sheet

| Method | Study Why | Maturity | Used In | Source |
|---|---|:---:|---|---|
| Sinusoidal PE | Original Transformer baseline | main | Early Transformer | [link](https://arxiv.org/abs/1706.03762) |
| RoPE | Modern decoder LLM default | main | LLaMA, Qwen, DeepSeek | [link](https://arxiv.org/abs/2104.09864) |
| YaRN | Efficient RoPE context extension | main | Long-context LLaMA/Qwen | [link](https://arxiv.org/abs/2309.00071) |
| LongRoPE | Non-uniform RoPE interpolation | main | Long-context research | [link](https://arxiv.org/abs/2402.13753) |
| LongRoPE2 | Continuation of LongRoPE | watchlist | Research | [link](https://arxiv.org/abs/2502.20082) |
| MRoPE | Multimodal spatial/temporal PE | main | Qwen2.5-VL family | [link](https://arxiv.org/abs/2502.13923) |
| TMRoPE | Time-aligned multimodal RoPE | main | Qwen2.5-Omni | [link](https://arxiv.org/abs/2503.20215) |
| Interleaved-MRoPE | Interleaved text-image-video PE | main | Qwen3-VL | [link](https://arxiv.org/abs/2511.21631) |
| Periodic RoPE / P-RoPE | Infinite-context PE direction | watchlist | Research | [link](https://arxiv.org/abs/2605.27980) |
| MHRoPE | Multi-head freq for multimodal | watchlist | Research | [link](https://arxiv.org/abs/2510.23095) |
| MRoPE-I | Interleaved multimodal RoPE variant | watchlist | Research | [link](https://arxiv.org/abs/2510.23095) |
| NoPE | Transformers without explicit PE | watchlist | Research direction | [link](https://arxiv.org/abs/2505.11199) |
| HoPE (Hyperbolic) | Hyperbolic geometry PE | watchlist | Research | [link](https://arxiv.org/abs/2509.05218) |
| HoPE (Hybrid VLM) | VLM / long-video PE | watchlist | Research | [link](https://arxiv.org/abs/2505.20444) |
| LazyAttention | Deferred PE for KV reuse | watchlist | Research | [link](https://arxiv.org/abs/2606.04302) |

### Core Takeaway

```text
RoPE gives position through rotation.
YaRN and LongRoPE stretch RoPE for long context.
MRoPE/TMRoPE extend position from text into image, video, and audio time.
```

> 📌 **Cross-ref:** [Full paper index](references/papers.md)


## 1.3 — Attention Mechanism

**Why:** Vanilla self-attention has O(n²) cost and huge KV cache. Real systems reduce both.

### Evolution

```text
Self-Attention → MHA → MQA → GQA → Local / Sliding Window → Local-Global
→ MLA → Sparse → NSA → Lightning Attention → Linear Attention / KDA
→ Multimodal Self-Attention / Cross-Attention
```

### Cheat Sheet

| Method | Problem Solved | Maturity | Used In | Source |
|---|---|:---:|---|---|
| MHA | Full attention baseline | main | Original Transformer / GPT-style | [link](https://arxiv.org/abs/1706.03762) |
| MQA | Reduces KV cache by sharing K/V | main | PaLM-era / efficient decoders | [link](https://arxiv.org/abs/1911.02150) |
| GQA | Balance MHA quality + MQA speed | main | LLaMA 3, Qwen, Gemma line | [link](https://arxiv.org/abs/2305.13245) |
| Local Attention | Reduces cost over long context | main | Gemma 3 local/global | [link](https://arxiv.org/abs/2503.19786) |
| MLA | Compresses KV cache into latent | main | DeepSeek-V3 | [link](https://arxiv.org/abs/2412.19437) |
| Sparse Attention | Skips irrelevant tokens/blocks | main | DeepSeek sparse direction | [link](https://arxiv.org/abs/2412.19437) |
| NSA | Sparse long-context + selection | main | Research | [link](https://arxiv.org/abs/2502.11089) |
| Lightning Attention | Efficient long-context hybrid | main | MiniMax-01 / MiniMax-M1 | [link](https://arxiv.org/abs/2501.08313) · [M1](https://arxiv.org/abs/2506.13585) |
| KDA | Linear-attention fixed-state memory | main | Kimi Linear | [link](https://arxiv.org/abs/2510.26692) |
| Multimodal Self-Attention | Text/image/video reason together | main | Qwen3-VL | [link](https://arxiv.org/abs/2511.21631) |

### Core Takeaway

```text
GQA reduces KV heads.
MLA compresses KV cache.
Sparse attention skips tokens.
Linear / KDA replaces growing KV cache with fixed recurrent state.
```


## 1.4 — FFN / MoE

**Why:** Scaling dense FFNs gets expensive. MoE decouples **total params** from **active params per token**.

### Evolution

```text
Dense FFN → GELU FFN → Gated FFN → SwiGLU
→ Sparse MoE → Top-K Routing → Shared Experts → Fine-Grained Experts
→ Aux-loss-free Load Balancing → High-Sparsity MoE → Adaptive Routing
→ SeqTopK → SoftMoE → Sub-MoE / Expert Merging
→ PEER (Million tiny experts) → Memory Layers / MoD
```

### Cheat Sheet

| Method | Core Idea | Maturity | Used In | Source |
|---|---|:---:|---|---|
| Dense FFN | Same MLP for every token | main | GPT / LLaMA / Gemma dense | [link](https://arxiv.org/abs/1706.03762) |
| SwiGLU | Gated FFN activation | main | LLaMA-style, Kimi K2 | [link](https://arxiv.org/abs/2002.05202) |
| Sparse MoE | Many FFNs, activate few experts | main | Mixtral, DeepSeek, Qwen MoE, Llama 4 | [link](https://arxiv.org/abs/2412.19437) |
| DeepSeekMoE | Shared + fine-grained experts | main | DeepSeek-V2/V3/R1 | [link](https://arxiv.org/abs/2401.06066) |
| Aux-loss-free Balancing | Balances experts without strong aux loss | main | DeepSeek-V3 | [link](https://arxiv.org/abs/2412.19437) |
| High-Sparsity MoE | Huge total, low active params | main | Qwen3 MoE / Qwen3-Next | [link](https://arxiv.org/abs/2505.09388) |
| Adaptive Routing (Ada-K) | Token-budget routing | watchlist | Research | [link](https://arxiv.org/abs/2410.10456) |
| SeqTopK | Sequence-level expert budget | watchlist | Research | [link](https://arxiv.org/abs/2511.06494) |
| SoftMoE | Differentiable expert routing | watchlist | Research | latest papers |
| Expert-Choice Routing | Experts pick tokens | watchlist | Research | Expert Choice Routing papers |
| Sub-MoE / Expert Merging | Merge experts for deployment | watchlist | Research | [link](https://arxiv.org/abs/2506.23266) |
| PEER | Many tiny retrieved experts | watchlist | Research | [link](https://arxiv.org/abs/2407.04153) |
| Mixture-of-Depths | Route tokens by compute depth | watchlist | Research | [link](https://arxiv.org/abs/2404.02258) |
| MoE++ | Cheap zero/copy/constant experts | watchlist | Research | [link](https://arxiv.org/abs/2410.07348) |

### Core Takeaway

```text
Dense FFN gives every token the same network.
MoE gives each token a selected set of expert networks.
Modern MoE focuses on routing, balancing, and reducing active compute.
```


## 1.5 — Beyond Transformer

**The question:** Can we replace the transformer with linear-time or recurrent architectures?

### Evolution

```text
S4 / SSM → Mamba → Mamba-2 / SSD → Jamba → Nemotron-H
→ RWKV → RetNet → Kimi Linear → Titans → Vision Mamba / MambaVision
```

### Cheat Sheet

| Architecture | Core Idea | Maturity | Source |
|---|---|:---:|---|
| Mamba | Selective state-space sequence model | main | [link](https://arxiv.org/abs/2312.00752) |
| Mamba-2 | SSM-attention State Space Duality | main | [link](https://arxiv.org/abs/2405.21060) |
| Jamba | Hybrid Transformer + Mamba + MoE | main | [link](https://arxiv.org/abs/2403.19887) |
| Nemotron-H | Hybrid Mamba-Transformer LLM | main | [link](https://arxiv.org/abs/2504.03624) |
| RWKV-7 | RNN-like LLM, constant memory/token | main | [link](https://arxiv.org/abs/2503.14456) |
| RetNet | Parallel training + recurrent inference | main | [link](https://arxiv.org/abs/2307.08621) |
| Kimi Linear | KDA + MLA hybrid efficient attention | main | [link](https://arxiv.org/abs/2510.26692) |
| Titans | Neural long-term memory | main | [link](https://arxiv.org/abs/2501.00663) |
| Gated DeltaNet | Linear/recurrent attention | watchlist | Qwen3-Next / Kimi reports |
| ATLAS / DeepTransformers | Test-time deep memory | watchlist | [link](https://arxiv.org/abs/2505.23735) |
| ModRWKV | Multimodal RWKV | watchlist | [link](https://arxiv.org/abs/2505.14505) |
| VMamba | Vision SSM (2D selective scan) | main | [link](https://arxiv.org/abs/2401.10166) |
| Vision Mamba / Vim | Mamba for visual patches | main | [link](https://arxiv.org/abs/2401.09417) |
| MambaVision | Hybrid Mamba-Transformer vision | main | [link](https://arxiv.org/abs/2407.08083) |
| xLSTM | Modern recurrent LLM (LSTM scaled) | watchlist | [link](https://arxiv.org/abs/2405.04517) |

### Core Takeaway

```text
Transformer remembers using KV cache.
Mamba / RWKV / RetNet / KDA remember using recurrent or fixed-size state.
Titans adds trainable long-term memory.
```


## 1.6 — Multimodal Architecture (Quick View)

**The question:** How do we go from text-only LLMs to models that see, hear, and speak?

Five patterns (detailed in Section 3):

```text
1. Dual Encoder (CLIP, SigLIP)            — image-text similarity only
2. Vision Encoder + Projector + LLM        — LLaVA pattern (chat on images)
3. Q-Former / Learned Query Bridge         — BLIP-2 (token-budget aware)
4. Perceiver Resampler                    — Flamingo-style (fixed visual tokens)
5. Unified / Interleaved Multimodal Tokens — Qwen-VL / Qwen3-VL / Qwen-Omni
```

See **Section 3 — VLM & Multimodal** for the full deep-dive.


## 1.7 — Model-Wise Architecture Map

> **Canonical, deduplicated model map.** No longer repeated in later sections.

| Model / Family | Architecture Tags | Source |
|---|---|---|
| Qwen3 | Dense + MoE, thinking/non-thinking, multilingual | [link](https://arxiv.org/abs/2505.09388) |
| Qwen2.5-VL | Dynamic resolution, window attention, absolute time | [link](https://arxiv.org/abs/2502.13923) |
| Qwen2.5-Omni | TMRoPE, Thinker-Talker | [link](https://arxiv.org/abs/2503.20215) |
| Qwen3-VL | Interleaved-MRoPE, DeepStack, dense + MoE VLM | [link](https://arxiv.org/abs/2511.21631) |
| DeepSeek-V3 | MLA, DeepSeekMoE, aux-loss-free load balancing | [link](https://arxiv.org/abs/2412.19437) |
| DeepSeek-R1 | RL reasoning, R1-Zero, distillation | [link](https://arxiv.org/abs/2501.12948) |
| Kimi Linear | KDA, linear attention, MLA hybrid | [link](https://arxiv.org/abs/2510.26692) |
| MiniMax-01 | Lightning attention, long-context VLM | [link](https://arxiv.org/abs/2501.08313) |
| MiniMax-M1 | Lightning attention, hybrid MoE, long-context reasoning | [link](https://arxiv.org/abs/2506.13585) |
| Gemma 3 | Multimodal open model, local/global attention | [link](https://arxiv.org/abs/2503.19786) |
| Llama 4 | MoE, multimodal, long context, iRoPE direction | [blog](https://ai.meta.com/blog/llama-4-multimodal-intelligence/) |
| Jamba | Transformer + Mamba + MoE | [link](https://arxiv.org/abs/2403.19887) |
| Nemotron-H | Hybrid Mamba-Transformer | [link](https://arxiv.org/abs/2504.03624) |
| RWKV-7 | Modern recurrent LLM | [link](https://arxiv.org/abs/2503.14456) |
| Titans | Neural long-term memory | [link](https://arxiv.org/abs/2501.00663) |

### Closed-Source Tracking

| Model Family | Use For | Architecture Disclosure |
|---|---|---|
| OpenAI GPT-5 / GPT-5.x | System card, behavior, routing/reasoning | Low |
| Claude 4 / Opus / Sonnet | System card, tool use, extended thinking | Low |
| Gemini 2.5 / Gemini 3 | Multimodal reasoning, long context | Low |
| Grok 4 / 4.x | Product behavior, tool use | Low |

**Rule:** Use closed-source docs for system behavior. Use open/open-weight reports for real architecture study.


## 1.8 — Decision Tree & Production Rule

### Which Architecture Should I Care About?

```text
Just want to use a model?
    → Read "Cheat Sheet" rows marked main, focus on GQA + SwiGLU + RoPE

Training your own LLM?
    → Add MLA + DeepSeekMoE + auxiliary-loss-free balancing

Long context (1M+ tokens)?
    → Add YaRN / LongRoPE / Lightning Attention / Linear Attention (KDA)

Multimodal?
    → Qwen2.5-VL / Qwen3-VL architecture patterns + MRoPE / TMRoPE

Pushing inference cost down?
    → Read Section 2 (Attention & Serving Kernels)

Curious about what's next?
    → Watchlist items (NoPE, P-RoPE, HoPE, xLSTM, Gated DeltaNet)
```

### Production Rule

```text
Architecture tells you what the model is.
It does NOT tell you how fast or cheap it runs.
For cost/latency, jump to Section 2.
```

---

## 📚 Section 1 Summary

```text
PE:      RoPE family → YaRN / LongRoPE → multimodal RoPE
Attn:    MHA → GQA → MLA → KDA / Lightning / Sparse
FFN:     Dense FFN → SwiGLU → MoE (DeepSeekMoE / High-sparsity)
Beyond:  Transformer → Mamba / RWKV / RetNet / Kimi Linear / Titans
Models:  Qwen, DeepSeek, Kimi, MiniMax, Gemma, Llama, Jamba, RWKV
```


# Section 2 — Attention & Serving Kernels

> **What this section is for:** Understand the **kernels, memory managers, and schedulers** that make LLMs run fast and cheap in production. These are NOT new model architectures — they are runtime optimizations.


## 2.0 — The Three Buckets (Don't Mix Them)

Most confusion in this space comes from mixing these up:

```text
Model architecture attention:
  MHA, MQA, GQA, MLA, sparse attention, local-global attention, KDA

Attention kernel optimization:
  FlashAttention, SageAttention, FlashInfer, FlexAttention

KV-cache / serving optimization:
  PagedAttention, vAttention, RadixAttention, prefix caching, chunked prefill

Distributed long-context systems:
  RingAttention, DeepSpeed-Ulysses, USP
```


## 2.1 — Attention Kernels

| Method | Bucket | What It Solves | Source |
|---|---|---|---|
| FlashAttention | Kernel | Computes exact attention faster by reducing HBM memory traffic | [link](https://arxiv.org/abs/2205.14135) |
| FlashAttention-2 | Kernel | Better parallelism and work partitioning | [link](https://arxiv.org/abs/2307.08691) |
| FlashAttention-3 | Kernel | Hopper/H100 async + FP8 attention | [link](https://arxiv.org/abs/2407.08608) |
| xFormers | Library | Practical attention backend in PyTorch ecosystem | [link](https://github.com/facebookresearch/xformers) |
| SageAttention | Quantized kernel | 8-bit attention acceleration | [link](https://arxiv.org/abs/2410.02367) |
| SageAttention2 | Quantized kernel | INT4/FP8 with smoothing/outlier handling | [link](https://arxiv.org/abs/2411.10958) |
| SageAttention3 | Quantized kernel | FP4 attention for Blackwell-era GPUs | [link](https://arxiv.org/abs/2505.11594) |
| FlashInfer | Inference engine | Optimized attention kernels and runtime pieces | [link](https://arxiv.org/abs/2501.01005) |
| FlexAttention | Programmable | Express attention variants, get optimized kernels | [link](https://arxiv.org/abs/2412.05496) |
| FlashMLA | MLA-specific runtime | Optimized serving for MLA-style attention | watchlist |

### Decision Tree

```text
Default attention backend?
    → FlashAttention-2 / FlashAttention-3

H100 / Hopper GPU?
    → FlashAttention-3 (FP8)

Blackwell GPU?
    → SageAttention3 (FP4, watchlist)

Need custom attention pattern?
    → FlexAttention

Need MLA-specific kernel?
    → FlashMLA (watchlist)
```


## 2.2 — KV-Cache & Serving Optimization

| Method | Bucket | What It Solves | Source |
|---|---|---|---|
| PagedAttention | KV-cache memory | Manages KV cache in blocks to reduce fragmentation | [link](https://arxiv.org/abs/2309.06180) |
| vAttention | KV-cache virtual memory | Virtual memory to manage KV cache with logical contiguity | [link](https://arxiv.org/abs/2405.04437) |
| RadixAttention | Prefix/KV reuse | Reuses KV cache for shared prompt prefixes | [link](https://arxiv.org/abs/2312.07104) |
| Prefix Caching | Serving | Reuses precomputed KV for repeated prompts | [vLLM docs](https://docs.vllm.ai/en/latest/automatic_prefix_caching/details.html) |
| Chunked Prefill | Scheduling | Splits long prompt prefill into chunks | [vLLM docs](https://docs.vllm.ai/en/latest/performance/optimization.html) |
| Continuous Batching | Scheduling | Dynamically batches requests during decoding | [link](https://arxiv.org/abs/2309.06180) |
| KV Cache Quantization | Memory | Reduces KV-cache footprint with lower precision | [link](https://arxiv.org/abs/2401.18079) |
| KVTuner | Quantization | Layer-wise mixed precision KV quantization | [link](https://arxiv.org/abs/2502.04420) |
| KVLinC | Quantization | Hadamard rotation + linear correction | [link](https://arxiv.org/abs/2510.05373) |
| TurboQuant | Quantization | Extreme KV/vector compression | watchlist |
| Disaggregated Prefill/Decode | Architecture | Separates prefill-heavy and decode-heavy workloads | vLLM / SGLang / TensorRT-LLM docs |
| Speculative Decoding | Decode accel | Draft model predicts, target verifies | Various serving docs |
| KV Offloading | Memory | Moves KV/cache/state to CPU/SSD where needed | FlexGen / InstInfer papers |

### Decision Tree

```text
OOM on long context?
    → PagedAttention (default in vLLM) → vAttention → KV Quantization → KV Offloading

Many requests with same system prompt?
    → Prefix Caching + RadixAttention

Mixed prefill + decode traffic?
    → Disaggregated Prefill/Decode + Continuous Batching

Need 2–4x decode speedup?
    → Speculative Decoding
```

> 📌 **Runnable example:** [`examples/04_serving/vllm_config.py`](examples/04_serving/vllm_config.py) · [`examples/04_serving/kserve_manifest.yaml`](examples/04_serving/kserve_manifest.yaml)


## 2.3 — Distributed Long-Context Attention

| Method | What It Solves | Source |
|---|---|---|
| RingAttention | Splits sequence attention across devices in a ring | [link](https://arxiv.org/abs/2310.01889) |
| DeepSpeed-Ulysses | Long-context training through sequence parallelism | [link](https://arxiv.org/abs/2309.14509) |
| USP | Combines sequence parallel strategies for long context | [link](https://arxiv.org/abs/2405.07719) |


## 2.4 — Architecture vs Kernel vs Serving — Cheat Sheet

| Method | Changes Model Architecture? | Main Layer |
|---|:---:|---|
| GQA | Yes | Model attention |
| MLA | Yes | Model attention |
| KDA / Linear Attention | Yes | Model attention / recurrent state |
| FlashAttention | No | Kernel |
| SageAttention | No | Quantized kernel |
| PagedAttention | No | KV-cache memory manager |
| vAttention | No | KV-cache virtual memory manager |
| RadixAttention | No | Prefix/KV-cache reuse |
| RingAttention | System-level | Distributed long-context attention |


## 2.5 — Production Rule

```text
Architecture methods decide what the model is.
Serving methods decide how cheaply and reliably the model runs in production.
```

For 90% of production deployments, you don't need to invent a new kernel — you need to:

1. **Use vLLM or TGI** (they bundle PagedAttention + Continuous Batching + FlashAttention)
2. **Enable prefix caching** if you have system prompts or RAG
3. **Quantize** the model (INT8 or INT4) if VRAM is tight
4. **Use speculative decoding** if decode latency matters
5. **Move to disaggregated serving** only if prefill is starving decode

---

## 📚 Section 2 Summary

```text
Kernels:   FlashAttention-2/3 (default) · SageAttention (quantized) · FlexAttention (custom)
Memory:    PagedAttention · vAttention · RadixAttention · KV quantization · KV offload
Schedul:   Prefix caching · Chunked prefill · Continuous batching · Disaggregated prefill/decode
Speedup:   Speculative decoding (2–4x decode)
Long ctx:  RingAttention · Ulysses · USP
```


# Section 3 — VLM & Multimodal

> **What this section is for:** Choose the right architecture pattern when extending LLMs to see, hear, and speak. Covers vision encoders, bridges, multimodal embeddings, and document understanding.


## 3.0 — Why VLMs Are Not Just "LLM + Image Input"

A VLM needs:

```text
1. A vision encoder (usually ViT-style)
2. A bridge / projector to map vision features into LLM token space
3. Multimodal position encoding (MRoPE / TMRoPE)
4. Multimodal instruction data
5. Special tokenization for image / video / audio
```

### Architecture Evolution

```text
CNN / ViT
↓
CLIP / ALIGN
↓
BLIP
↓
BLIP-2 + Q-Former
↓
LLaVA / MiniGPT-4
↓
InternVL / Qwen-VL
↓
Qwen-Omni / Omni models
↓
Unified multimodal / native omni architectures
```


## 3.1 — Five Architecture Patterns

### Pattern 1 — Dual Encoder

```text
Image → Vision Encoder → Image Embedding
Text  → Text Encoder   → Text Embedding
Image/Text similarity through contrastive loss
```

**Best for:** image retrieval, text-image matching, zero-shot classification
**Not best for:** visual reasoning, long visual QA, multi-turn image chat
**Examples:** CLIP, OpenCLIP, ALIGN, SigLIP

### Pattern 2 — Vision Encoder + Linear Projector + LLM

```text
Image
→ CLIP / ViT vision encoder
→ linear projector / MLP projector
→ image tokens in LLM embedding space
→ LLM decoder generates answer
```

**Best for:** visual chat, document QA, image reasoning
**Limitation:** projector may lose fine-grained visual information
**Examples:** LLaVA, MiniGPT-4, early open VLMs

### Pattern 3 — Q-Former / Learned Query Bridge

```text
Image patches
→ Vision encoder
→ Q-Former learned queries attend to image features
→ compressed visual tokens
→ LLM
```

**Best for:** token-budget-constrained VLMs
**Examples:** BLIP-2, InstructBLIP

### Pattern 4 — Perceiver Resampler

```text
Image features
→ learned latent queries
→ fixed number of visual tokens
→ language model with cross-attention
```

**Examples:** Flamingo-style architectures

### Pattern 5 — Unified / Interleaved Multimodal Token Architecture

```text
Text tokens + image tokens + video tokens + audio tokens
→ unified model context
→ multimodal attention
→ text/audio/image/video response
```

**Best for:** deep cross-modal reasoning (video + timestamp + text + speech)
**Examples:** Qwen-VL, Qwen3-VL, Qwen-Omni, Gemini-style, GPT-4o-style


## 3.2 — VLM Technical Report Library

| Model / Family | Pattern | What to Study | Source |
|---|---|---|---|
| CLIP | Dual encoder | Contrastive image-text embedding | OpenAI paper |
| OpenCLIP | Dual encoder | Open reproduction / scaling | OpenCLIP repo |
| SigLIP | Dual encoder | Sigmoid loss for image-text pretraining | SigLIP paper |
| BLIP | VL pretraining | Captioning + filtering + bootstrapping | Salesforce BLIP |
| BLIP-2 | Vision + Q-Former + LLM | Frozen vision/LLM, trainable bridge | [link](https://arxiv.org/abs/2301.12597) |
| LLaVA | CLIP + projector + LLaMA | Two-stage visual instruction tuning | [LLaVA](https://llava-vl.github.io/) |
| InternVL | Large vision encoder + LLM bridge | Scaled vision foundation | [link](https://arxiv.org/abs/2312.14238) |
| Qwen2.5-VL | Modern VLM | Dynamic resolution, window attention, absolute time | [link](https://arxiv.org/abs/2502.13923) |
| Qwen2.5-Omni | Omni model | TMRoPE, Thinker-Talker | [link](https://arxiv.org/abs/2503.20215) |
| Qwen3-VL | Modern VLM | Interleaved-MRoPE, DeepStack, dense + MoE VLM | [link](https://arxiv.org/abs/2511.21631) |
| Gemma 3 multimodal | Open multimodal | Local/global attention | [link](https://arxiv.org/abs/2503.19786) |
| MiniCPM-V | Efficient VLM | Small-device / efficient VLM | OpenBMB |
| GPT-4o / Gemini | Closed omni | Behavior and product direction only | Limited disclosure |


## 3.3 — Multimodal Embeddings

| Type | Meaning | Examples |
|---|---|---|
| Dense text embedding | One vector per text / chunk | OpenAI embeddings, BGE, Jina |
| Dense image embedding | One vector per image | CLIP, SigLIP, OpenCLIP |
| Cross-modal embedding | Image and text in same space | CLIP, SigLIP |
| Sparse embedding | Lexical / token-weighted representation | BM25, SPLADE-style |
| Multi-vector embedding | Multiple vectors per document/page/image | ColBERT, ColPali |
| Multimodal embedding | Text + image + layout-aware embedding | ColPali, multimodal retrievers |

### ColPali — Multi-vector Document Embedding

Instead of OCR-ing a page first:

```text
PDF page / image
→ vision-language encoder
→ multiple page-level visual/text vectors
→ late interaction retrieval
```

**Why useful:** tables, layout, charts, and visual structure can be retrieved without OCR-only loss.

### Core Takeaway

```text
CLIP/SigLIP are strong for image-text similarity.
BGE/Jina/NV-Embed are strong for text retrieval.
ColPali-style models are strong for document/page retrieval because they preserve visual layout.
```


## 3.4 — Decision Tree & Production Rule

### Which VLM Pattern?

```text
Image-text similarity / retrieval only?
    → Dual Encoder (CLIP / SigLIP)

Visual chat on natural images?
    → Vision Encoder + Projector + LLM (LLaVA pattern)

Limited token budget / many image patches?
    → Q-Former / Perceiver

Document understanding (PDFs, charts)?
    → ColPali-style multi-vector retrieval

Real-time multimodal interaction (text + audio + video)?
    → Unified / Interleaved (Qwen-VL / Qwen-Omni)
```

### Production Rule

```text
Most enterprise VLMs in 2025–2026 use Pattern 2 (projector + LLM) for chat
or Pattern 5 (unified tokens) for advanced multimodal reasoning.
For document QA, prefer ColPali-style multi-vector retrieval over text-only OCR pipelines.
```

---

## 📚 Section 3 Summary

```text
Pattern 1: Dual Encoder (CLIP, SigLIP)               — retrieval only
Pattern 2: Vision + Projector + LLM (LLaVA)          — visual chat
Pattern 3: Q-Former bridge (BLIP-2)                  — token-budget aware
Pattern 4: Perceiver resampler (Flamingo)            — fixed visual tokens
Pattern 5: Unified multimodal (Qwen-VL / Omni)       — full omni reasoning
Document:  ColPali-style multi-vector (no OCR loss)
```


# Section 4 — Coding Models

> **What this section is for:** Understand why coding models beat general LLMs at code, and pick the right one for IDE completion vs. agentic coding.


## 4.0 — Why Coding Models Differ

**Common misunderstanding:** `Coding model ≠ normal LLM that knows Python`.

A coding model is improved through:

```text
1. More code-heavy pretraining
2. Repository-level data
3. Multi-language code data
4. Code-specific instruction tuning
5. Infilling / fill-in-the-middle training
6. Longer context
7. Tool-use / agentic coding data
8. Compiler/test feedback
9. Bug fixing and code repair datasets
```

### What Actually Improves Coding Capability

| Factor | Why it matters |
|---|---|
| Code-heavy tokens | Model sees syntax, libraries, APIs, patterns |
| Multi-file repo data | Learns imports, project structure, dependencies |
| Fill-in-the-middle | Better IDE-style completion |
| Long context | Can read more of a codebase |
| Test/repair data | Better debugging and patching |
| Agentic data | Better tool use, edit-run-debug loops |
| Code tokenizer | Better compression of symbols/indentation/API names |

**Key insight:** Most coding improvements come from **data + tokenizer + context length + instruction tuning + tool-use training + evaluation loop** — NOT from a radically different transformer architecture.


## 4.1 — Model Timeline & Library

```text
CodeGen → StarCoder → Code Llama → DeepSeek Coder → Qwen Coder
→ Codestral / Devstral → Kimi K2 → Tool-use coding agents
```

| Model | Why important | Source |
|---|---|---|
| Code Llama | Llama 2 continued on 500B code tokens | [blog](https://huggingface.co/blog/codellama) |
| StarCoder | 15B on permissively licensed GitHub, 80+ languages | [blog](https://huggingface.co/blog/starcoder) |
| DeepSeek Coder | Strong open coding family, math/code reasoning | [GitHub](https://github.com/deepseek-ai/DeepSeek-Coder) |
| Qwen2.5-Coder | Strong open-source coding model, Qwen family | [blog](https://www.alibabacloud.com/blog/qwen2-5-coder-series-powerful-diverse-practical_601765) |
| Codestral | Mistral coding model family | [news](https://mistral.ai/news/codestral) |
| Devstral | Agentic coding / software engineering direction | [news](https://mistral.ai/news/devstral) |
| Kimi K2 | Agentic / tool-use coding direction | Moonshot official |


## 4.2 — Evaluation Benchmarks

| Benchmark | Measures |
|---|---|
| HumanEval | Function-level code generation |
| MBPP | Basic Python programming |
| EvalPlus | More rigorous HumanEval/MBPP tests |
| LiveCodeBench | Recent coding tasks |
| SWE-bench | Real GitHub issue resolution |
| Aider benchmark | Code repair / agentic coding workflow |
| BigCodeBench | Practical code generation |
| RepoBench | Repository-level understanding |

### Core Takeaway

```text
HumanEval checks function generation.
SWE-bench checks software engineering.
A model can be good at HumanEval but weak at repo-level debugging.
```


## 4.3 — Decision Tree & Production Rule

### Which Coding Model?

```text
Need an IDE coding assistant?
    → Qwen2.5-Coder / Codestral / DeepSeek Coder

Need agentic coding (read repo, edit, run tests)?
    → Devstral / Kimi K2 / DeepSeek Coder + tools

Need local / offline coding model?
    → Qwen2.5-Coder (1.5B / 7B)

Need best function-level completion?
    → Codestral / Code Llama

Best open reasoning + coding?
    → DeepSeek Coder / Qwen2.5-Coder-32B
```

### Production Rule

```text
For agentic coding, the model is only 30% of the system.
The other 70% is:
  - tool framework (file read, grep, shell, tests)
  - sandbox (no accidental file deletion)
  - retry + repair loop
  - evaluation harness (does the patch pass tests?)
```

---

## 📚 Section 4 Summary

```text
Coding improvements = data + tokenizer + long context + tool-use training + eval loop
Models: Qwen2.5-Coder, Codestral, Devstral, DeepSeek Coder, StarCoder, Kimi K2
Benchmarks: HumanEval (function) vs SWE-bench (real issues) — measure BOTH
Agentic coding needs model + tools + sandbox + eval, not just the model
```


# Section 5 — Fine-Tuning & PEFT

> **What this section is for:** Choose the right fine-tuning method (full FT, LoRA, QLoRA, DoRA) for your compute budget and quality target. Includes capacity planning and framework selection.


## 5.0 — Why Fine-Tuning Exists

Base model knows general knowledge. Fine-tuning adapts it to:

```text
Domain style
Instruction following
Specific tasks
Reasoning traces
Tool-calling format
Safety behavior
Enterprise data style
Coding conventions
VLM instruction behavior
```

### Method Evolution

```text
Full Fine-Tuning
↓
Adapters
↓
Prompt Tuning
↓
Prefix Tuning
↓
LoRA
↓
QLoRA
↓
DoRA
↓
AdaLoRA / modern PEFT variants
↓
RLHF / Preference Optimization
```


## 5.1 — Fine-Tuning Categories

| Category | What changes | Cost |
|---|---|---|
| Full fine-tuning | All model weights | Highest |
| SFT | Usually all weights or adapters on instruction data | Medium to high |
| PEFT | Small trainable modules | Low |
| Preference optimization | Aligns preferred outputs | Medium |
| Continued pretraining | More tokens on domain corpus | Very high |
| VLM fine-tuning | Vision bridge / projector / LLM / vision encoder | Varies |


## 5.2 — PEFT Methods Comparison

| Method | How it works | Best for | Limitations |
|---|---|---|---|
| Adapter tuning | Inserts small trainable layers | Modular task adaptation | Adds inference path |
| Prompt tuning | Learns soft prompt embeddings | Very low parameter tuning | Lower capacity |
| Prefix tuning | Learns prefix key/value vectors | Generation control | Can be weaker than LoRA |
| **LoRA** | Adds low-rank matrices to attention/MLP weights | Most common PEFT | Rank choice matters |
| **QLoRA** | Quantizes base model to 4-bit and trains LoRA | Large model on limited GPU | Slower due to quant/dequant |
| **DoRA** | Decomposes weight into magnitude + direction, applies LoRA to direction | Better LoRA quality | More complex/newer |
| AdaLoRA | Dynamically allocates rank budget | Efficient rank allocation | More moving parts |
| IA3 | Learns scaling vectors | Very parameter-efficient | Less flexible than LoRA |

### LoRA — the canonical PEFT method

```text
Base weight W is frozen.
Trainable low-rank update ΔW = A × B is added.
Only A and B are trained.
```

### QLoRA — LoRA on a quantized base

```text
Base model is loaded in 4-bit (NF4).
LoRA adapters are trained on top.
Allows 65B models on a single 48GB GPU.
```

### DoRA — better LoRA via decomposition

```text
Weight = magnitude × direction
DoRA keeps magnitude trainable
and applies LoRA-style low-rank update to direction only.
```

> 📌 **Runnable example:** [`examples/03_finetune/lora_sft.py`](examples/03_finetune/lora_sft.py)


## 5.3 — Decision Tree — Which Method Should I Use?

```text
Need domain knowledge from huge corpus?
    → Continued pretraining

Need instruction-following for task format?
    → SFT

Limited GPU?
    → LoRA or QLoRA

Very limited GPU?
    → QLoRA + Unsloth

Need best accuracy and have GPUs?
    → Full fine-tuning / FSDP / DeepSpeed

Need preference alignment?
    → DPO / ORPO / SimPO / GRPO  (see Section 6)

Need reasoning improvement?
    → SFT with reasoning traces + GRPO / rejection sampling / distillation

Need VLM adaptation?
    → Projector tuning → multimodal SFT → optional multimodal preference tuning
```

### Practical Rule

```text
Start with SFT + LoRA/QLoRA.
Use DPO/ORPO/SimPO only after you have preference pairs.
Use GRPO if you have verifiable rewards or reasoning tasks.
Use full fine-tuning only when PEFT is not enough and budget exists.
```


## 5.4 — Capacity Planning

### Memory Intuition

```text
FP32 = 4 bytes per parameter
FP16/BF16 = 2 bytes per parameter
INT8 = 1 byte per parameter
INT4/NF4 = 0.5 byte per parameter
```

But training needs more than weights:

```text
weights
+ gradients
+ optimizer states
+ activations
+ LoRA/adapters
```

### Capacity Table

| GPU VRAM | Practical LoRA | Practical QLoRA | Notes |
|---:|---|---|---|
| 12GB | 1B–3B comfortable; 7B possible with aggressive settings | 3B–7B possible | Demos, small labs |
| 16GB | 7B LoRA possible | 7B–13B QLoRA possible | Low-cost training |
| 24GB | 7B LoRA comfortable | 13B QLoRA possible | RTX 4090 class |
| 48GB | 13B–34B LoRA | 34B QLoRA possible | A6000 / L40S class |
| 80GB | 34B LoRA; 70B QLoRA possible | 70B QLoRA | A100/H100 class |
| Multi-GPU 80GB | 70B+ with FSDP/ZeRO | 100B+ with sharding | Production |

### Things That Increase Memory

```text
Long sequence length
Large batch size
Full fine-tuning
Vision tokens in VLM
RLHF with multiple models
PPO with policy + reference + reward + value model
High-resolution images
Video frames
```

### Things That Reduce Memory

```text
QLoRA
Gradient checkpointing
FlashAttention
FSDP / ZeRO
Gradient accumulation
Lower sequence length
Smaller batch size
Freeze vision encoder
Projector-only tuning
```


## 5.5 — Training Frameworks

| Framework | Best for | Not best for |
|---|---|---|
| Hugging Face Transformers | General training and model loading | Large distributed RLHF alone |
| PEFT | LoRA/QLoRA/adapters | Full RLHF pipeline |
| TRL | PPO, DPO, GRPO, reward modeling | Very large infra orchestration alone |
| Unsloth | Fast single-GPU LoRA/QLoRA/GRPO | Open multi-GPU at massive scale |
| Axolotl | YAML SFT/DPO/PPO/GRPO pipelines | Beginners who dislike config complexity |
| LLaMA-Factory | UI-friendly fine-tuning | Advanced distributed RLHF |
| DeepSpeed | ZeRO, large distributed training | Simple local tuning |
| FSDP | PyTorch-native model sharding | Beginners |
| Megatron-LM | Very large-scale pretraining | Small experiments |
| NeMo | NVIDIA enterprise training stack | Lightweight hobby training |
| Ray Train | Distributed orchestration | Model-specific tuning alone |
| OpenRLHF | RLHF training stack | Simple LoRA only |
| verl | RLHF/GRPO style training | Simple beginner LoRA |
| SkyPilot | Cloud orchestration | Training algorithm itself |

### Framework Decision Tree

```text
Single GPU LoRA?
    → Unsloth / PEFT / LLaMA-Factory

Multi-GPU SFT?
    → Axolotl / DeepSpeed / FSDP

DPO or PPO pipeline?
    → TRL / Axolotl / OpenRLHF

GRPO / reasoning RL?
    → verl / TRL / OpenRLHF / Unsloth for small setups

Enterprise NVIDIA stack?
    → NeMo / Megatron / TensorRT-LLM

Ray-based orchestration?
    → Ray Train + custom training scripts
```

---

## 📚 Section 5 Summary

```text
Start:  SFT + LoRA / QLoRA (lowest cost, fastest iteration)
Mid:    DPO / ORPO / SimPO once you have preference pairs
High:   Full fine-tuning when PEFT is not enough
Tools:  Unsloth (single GPU) / TRL (RLHF) / DeepSpeed / FSDP (distributed)
Capacity: 12GB → demos · 24GB → 7B LoRA · 48GB → 34B LoRA · 80GB → 70B QLoRA
```


# Section 6 — Alignment & RLHF

> **What this section is for:** Choose the right post-training method (DPO, ORPO, SimPO, GRPO) to align model behavior with preferences — without the complexity of PPO.


## 6.0 — Why Alignment Exists

A base model is a next-token predictor. It doesn't naturally:

- Refuse harmful requests
- Follow a specific output format
- Prefer concise over verbose answers
- Reason step-by-step
- Align with enterprise tone/style

Alignment techniques add these behaviors.

### Method Evolution

```text
SFT → Reward Model → PPO-based RLHF → RLAIF
→ DPO → IPO → KTO → ORPO → SimPO → GRPO
```


## 6.1 — Method Comparison

| Method | Needs reward model? | Needs reference model? | Online RL? | Best for | Weakness |
|---|:---:|:---:|:---:|---|---|
| PPO RLHF | Yes | Usually yes | Yes | Strong alignment with reward model | Complex, unstable, expensive |
| DPO | No explicit RM | Yes | No | Offline preference pairs | Sensitive to preference data quality |
| IPO | No explicit RM | Yes | No | Alternative preference optimization | Less common |
| KTO | No explicit RM | Usually no/varies | No | Binary desirable/undesirable feedback | Newer, less standard |
| ORPO | No | No | No | Simple SFT + preference in one stage | Objective tuning needed |
| SimPO | No | No | No | Reference-free preference training | Newer; needs validation |
| GRPO | Reward/verifier useful | No critic | Yes-ish / online sampling | Reasoning/verifiable tasks | More complex than DPO/ORPO |
| RLAIF | AI feedback instead of human | Varies | Varies | Scalable preference labels | Can amplify model bias |


## 6.2 — The Core Idea Behind Each Method

### Traditional RLHF (PPO)

```text
1. SFT model on demonstrations
2. Collect human preference pairs
3. Train reward model
4. Use PPO to optimize policy model against reward model
```

**Pros:** Powerful alignment method, can optimize behavior beyond supervised data
**Cons:** Complex, expensive, needs reward model, needs sampling loop, tuning can be unstable

### DPO — Direct Preference Optimization

```text
Use preference pairs directly.
Avoid explicit reward model and PPO loop.
```

### ORPO — Odds Ratio Preference Optimization

```text
Combine SFT and preference alignment into one objective.
No separate reference model.
```

### GRPO — Group Relative Policy Optimization

```text
Generate multiple responses for the same prompt.
Compare group rewards.
Avoid separate value critic used in PPO.
```

### SimPO — Simple Preference Optimization

```text
Use length-normalized log probability as an implicit reward.
Reference-free preference optimization.
```

> 📌 **Runnable example:** [`examples/03_finetune/dpo_example.py`](examples/03_finetune/dpo_example.py)


## 6.3 — Reasoning Model Training

### Common Training Stages

```text
Base LLM
↓
SFT on instruction data
↓
SFT on chain-of-thought / reasoning traces
↓
Preference optimization
↓
RL with verifiable rewards
↓
Distillation into smaller models
```

### Reward Types

| Reward type | Meaning | Use case |
|---|---|---|
| Outcome reward | Final answer is correct/incorrect | Math, code tests |
| Process reward | Intermediate reasoning steps are scored | Step-by-step reasoning |
| Rule-based reward | Programmatic verifier | Coding, math, structured tasks |
| Human preference reward | Human ranks outputs | Chat/helpfulness |
| AI feedback reward | Strong model ranks outputs | RLAIF |


## 6.4 — Decision Tree & Production Rule

### Which Method?

```text
Need basic tone/style alignment?
    → SFT + small DPO dataset

Need verifiable-task alignment (math/code)?
    → GRPO + rule-based verifier

Need a single unified stage?
    → ORPO

Need reference-free preference training?
    → SimPO

Already have a strong reward model?
    → PPO

Scaling preference labels without humans?
    → RLAIF (use a stronger model as judge)
```

### Practical Rule

```text
For enterprise chatbot: DPO/ORPO may be enough.
For math/code reasoning: GRPO/verifier-based RL becomes more relevant.
```

### Source Links

- DPO: https://arxiv.org/abs/2305.18290
- ORPO: https://arxiv.org/abs/2403.07691
- KTO: https://arxiv.org/abs/2402.01306
- GRPO / DeepSeekMath: https://arxiv.org/abs/2402.03300
- SimPO: https://arxiv.org/abs/2405.14734
- TRL: https://github.com/huggingface/trl

---

## 📚 Section 6 Summary

```text
PPO = powerful but expensive (reward model + sampling loop)
DPO = simpler offline preference optimization
ORPO/SimPO = reference-free simplification
GRPO = useful for reasoning when rewards/verifiers exist
Default path: SFT → DPO/ORPO → GRPO only for reasoning tasks
```


# Section 7 — RAGOps

> **What this section is for:** Connect LLMs to private / external knowledge with retrieval-augmented generation. Covers hybrid search, rerankers, multimodal RAG, and production reliability.


## 7.0 — Why RAG Exists

Base LLMs have three problems:

```text
1. Knowledge cutoff
2. Hallucination
3. No access to private enterprise data
```

RAG connects the LLM to external knowledge:

```text
User query → retrieve context → inject context → generate grounded answer
```

RAGOps is the discipline of running RAG **as a production system** — with monitoring, versioning, security, and feedback loops.

### Basic RAG Pipeline

```text
Documents
→ parsing / OCR
→ chunking
→ embedding
→ vector database
→ retrieval
→ prompt context
→ LLM answer
```


## 7.1 — Where Basic RAG Fails

| Failure | Meaning |
|---|---|
| Bad chunking | Important meaning split incorrectly |
| Weak embedding | Query and document do not match semantically |
| Low recall | Relevant chunk not retrieved |
| Low precision | Irrelevant chunks retrieved |
| No reranker | Best chunk not ranked at top |
| No metadata filter | Wrong user/tenant/source retrieved |
| Prompt injection | Document attacks the model |
| No eval | No proof answer is grounded |
| No index versioning | Cannot reproduce old behavior |


## 7.2 — Advanced RAG Components

| Component | Purpose |
|---|---|
| **Hybrid search** | BM25 keyword + dense vector retrieval |
| **Reranker** | Reorders retrieved chunks with stronger model |
| Query rewriting | Converts vague query into retriever-friendly query |
| Multi-query retrieval | Creates multiple query variants |
| **Metadata filtering** | Filters by user, tenant, source, date, permission |
| Contextual compression | Keeps only relevant context |
| Parent-child retrieval | Retrieves small chunks, returns parent document |
| GraphRAG | Uses entity/relation graph for reasoning |
| Multi-vector retrieval | Multiple vectors per document/page |
| Multimodal RAG | Retrieves images, tables, charts, PDFs, video/audio |

### Hybrid RAG

```text
BM25 / keyword search
+
dense vector search
+
score fusion
+
reranking
```

Dense retrieval catches meaning. BM25 catches exact words, IDs, names, legal clauses, product codes and error strings.

### Reranker

```text
Retriever = fast recall
Reranker = slower precision
```

Retriever gives candidates. Reranker deeply compares query + chunk and reorders top results.

### GraphRAG

GraphRAG helps when the answer needs relationships:

```text
Company → Project → Person → Contract → Risk → Decision
```

### Multimodal RAG

Use when documents are not pure text:

```text
PDF pages
tables
charts
screenshots
diagrams
images
videos
audio transcripts
```

> 📌 **Runnable example:** [`examples/01_rag/rag_basic.py`](examples/01_rag/rag_basic.py)


## 7.3 — Production RAGOps Pipeline

```text
Document upload
→ validation
→ malware / prompt injection scan
→ OCR / parsing
→ chunking
→ embedding
→ metadata tagging
→ vector index versioning
→ retrieval evaluation
→ deployment
→ monitoring
→ re-indexing
```


## 7.4 — RAGOps Metrics

| Layer | Metrics |
|---|---|
| Retrieval | Recall@K, Precision@K, MRR, nDCG |
| Reranking | Hit rate after rerank, MRR improvement |
| Generation | Faithfulness, groundedness, answer relevance |
| Safety | Injection resistance, PII leakage |
| System | Latency, cost, cache hit rate |
| Business | Task completion, user acceptance |


## 7.5 — Decision Tree & Production Rule

### Which RAG Pattern?

```text
Need exact IDs / legal clauses?
    → Hybrid search (BM25 + dense)

Need semantic answers?
    → Dense retrieval

Need better top-5 chunks?
    → Add a reranker

Need document layout / charts?
    → Multimodal RAG / ColPali-style retrieval

Need relationship reasoning?
    → GraphRAG

Need strict access control?
    → Metadata filters + tenant isolation

Need production reliability?
    → Eval + monitoring + index versioning
```

### Production Rule

```text
Basic RAG works for ~20% of real use cases.
For the remaining 80%, you need:
  - hybrid search (always, by default)
  - a reranker (Cross-encoder or ColBERT-style)
  - metadata filtering (for permission / multi-tenant)
  - eval harness (faithfulness + groundedness)
  - index versioning (so you can roll back)
  - injection scanning (on document upload)
```

---

## 📚 Section 7 Summary

```text
Pipeline: docs → parse → chunk → embed → index → retrieve → rerank → generate
Always:  hybrid search + reranker + metadata filters + eval harness
For docs: ColPali-style multi-vector (preserves layout)
For relations: GraphRAG
For safety: injection scan on ingest + faithfulness eval on output
```


# Section 8 — AgentOps

> **What this section is for:** Build agents that can plan, call tools, use memory, and recover from errors — without breaking in production.


## 8.0 — What is an AI Agent?

Simple chatbot:

```text
User → LLM → answer
```

Agent:

```text
User → planner → tool call → observation → reasoning → next tool → final answer
```

An agent can **reason, plan, call tools, use memory, observe results, continue steps, and return final answers**. But they're also fragile, expensive, and dangerous if not operated carefully.

### Agent vs Workflow

| System | Meaning | Use when |
|---|---|---|
| Workflow | Fixed sequence | Process is predictable |
| Agent | LLM decides next step | Task is open-ended |
| Multi-agent | Multiple specialized agents | Task needs role separation |
| Graph agent | State machine controls flow | Need reliability and control |


## 8.1 — Agentic Design Patterns

| Pattern | Meaning |
|---|---|
| ReAct | Reason + Act + Observe |
| Planner-executor | One component plans, another executes |
| Router | Routes task to specialist chain/agent |
| Reflection | Model critiques and revises |
| Tool-use agent | Calls APIs/functions |
| Human-in-loop | Human approval before risky action |
| Supervisor-worker | Supervisor coordinates agents |
| State machine agent | Graph controls allowed transitions |

### LangGraph-style Control

```text
State
→ Node
→ Conditional edge
→ Tool / Agent
→ Update state
→ Continue or stop
```

Graphs make agent behavior **inspectable, debuggable and recoverable**.

> 📌 **Runnable example:** [`examples/02_agent/langgraph_supervisor.py`](examples/02_agent/langgraph_supervisor.py)


## 8.2 — Tool Calling Requirements

```text
tool schema validation
input sanitization
timeout
retry
permission check
audit log
human approval for high-risk tools
```


## 8.3 — Agent Memory

| Memory type | Meaning |
|---|---|
| Short-term memory | Current session state |
| Long-term memory | Persisted facts/preferences |
| Episodic memory | Past interactions |
| Semantic memory | User/business knowledge |
| Tool memory | Previous tool outputs |
| Vector memory | Retrieved embedded memory |
| Structured memory | SQL/JSON profile/entity store |

### Memory Risks

```text
wrong memory
outdated memory
cross-user leakage
private data stored without consent
prompt injection stored as memory
malicious user teaches bad facts
```

### Safe Memory Design

```text
memory write should be explicit
sensitive memory should be blocked
memory should have source + timestamp
memory should be user-scoped
memory should be editable/deletable
retrieval should be filtered
```


## 8.4 — Multi-Agent Architecture

```text
User request
→ Supervisor agent
→ Research agent
→ Code agent
→ Critic agent
→ Safety agent
→ Final response
```

**Use multi-agent only when the task needs role separation. Do not use it for simple Q&A.**


## 8.5 — AgentOps Monitoring

Track:

```text
agent steps
tool calls
tool errors
latency per tool
cost per step
loop count
state transitions
memory reads/writes
failure reason
user feedback
```


## 8.6 — Decision Tree & Production Rule

### Which Agent Pattern?

```text
Simple Q&A or fixed sequence?
    → Plain chain / workflow

Need a model to choose next step?
    → ReAct agent

Need role separation?
    → Multi-agent (supervisor + workers)

Need reliability / audit trail?
    → State machine / LangGraph-style graph

Need user-specific memory?
    → Add memory layer (with safety controls)

Need tool access?
    → Tool-use agent (with sandbox + permissions)
```

### Production Rule

```text
Agent systems fail in three ways:
  1. Loop forever (no max steps)
  2. Call unsafe tools (no permission check)
  3. Hallucinate tool outputs (no schema validation)

Production-grade agents must enforce:
  - max steps + timeout
  - tool allow-list + permission check
  - schema validation + sandbox
  - audit log + user feedback
  - rollback to last good state
```

---

## 📚 Section 8 Summary

```text
Workflow:    fixed sequence (predictable)
ReAct:       model picks next step (open-ended)
Multi-agent: supervisor + workers (role separation)
Graph:       state machine controls flow (reliability)
Memory:      user-scoped, source-tagged, deletable (safety)
Tools:       allow-list + permissions + audit log + sandbox (safety)
Always:      max steps + timeout + audit log + rollback
```


# Section 9 — LLMOps & EvalOps

> **What this section is for:** Treat prompts, adapters, and models as versioned artifacts with eval gates, observability, and rollback — like any other production software.


## 9.0 — Why LLMOps Exists

LLM apps deploy a **system**, not only a model:

```text
model
prompt
retriever
tools
memory
guardrails
evaluation
serving layer
user feedback
```

Without LLMOps discipline, you cannot:
- Roll back a bad prompt change
- Prove answer quality is improving
- Detect regressions
- Reproduce past behavior
- Audit decisions


## 9.1 — PromptOps

PromptOps manages prompts like software artifacts:

```text
prompt template
prompt version
test set
approval
deployment
rollback
A/B test
monitoring
```

### Prompt Registry Example

```json
{
  "prompt_id": "rag_answer_v3",
  "version": "3.2.1",
  "owner": "ai-platform",
  "model": "qwen-72b",
  "temperature": 0.2,
  "status": "production"
}
```


## 9.2 — EvalOps

| Eval type | Purpose |
|---|---|
| Golden dataset eval | Compare against expected answers |
| LLM-as-judge | Judge quality using another model |
| Human eval | Expert scoring |
| RAG faithfulness eval | Answer grounded in retrieved context |
| Tool-call eval | Correct tool and arguments |
| Safety eval | Jailbreak, PII, toxicity |
| Regression eval | New version not worse than old |
| Cost-latency eval | Production feasibility |

> 📌 **Runnable example:** [`examples/06_eval/llm_as_judge.py`](examples/06_eval/llm_as_judge.py)


## 9.3 — Observability

Track:

```text
prompt
model
retrieved chunks
tool calls
latency
token usage
cost
error
fallback
user feedback
trace ID
```


## 9.4 — Version Everything

```text
model version
adapter version
prompt version
retriever/index version
embedding model version
reranker version
tool schema version
guardrail version
```

### Why?

```text
Without version pinning, you cannot:
  - reproduce a past good answer
  - roll back a regression
  - blame a specific change for a quality drop
  - prove to auditors that nothing unsafe changed
```


## 9.5 — Decision Tree & Production Rule

### Which Eval Pattern?

```text
Need to A/B test prompts?
    → Prompt registry + traffic split

Need to catch regressions automatically?
    → Golden dataset eval + CI gate

Need qualitative feedback?
    → LLM-as-judge on sampled traffic

Need real-world quality signal?
    → User feedback + thumbs up/down + outcome tracking

Need end-to-end traceability?
    → Distributed tracing (OpenTelemetry) + trace IDs
```

### Production Rule

```text
Treat prompts and adapters as versioned artifacts.
Run eval suite on every change.
Block deploys that fail the suite.
Track business metrics (acceptance, completion) alongside technical metrics.
```

---

## 📚 Section 9 Summary

```text
Prompts:    version + test + approval + deploy + rollback (treat as code)
Eval:       golden dataset + LLM-as-judge + regression + safety
Observe:    trace ID + latency + cost + error + feedback (always)
Version:    model + adapter + prompt + index + embedding + tool schema + guardrail
Block:      deploys that fail the eval suite
```


# Section 10 — AWS Production Architecture

> **What this section is for:** Pick the right AWS service for each layer (Bedrock vs EKS vs SageMaker vs Ray) and avoid months of misaligned platform work.


## 10.0 — Managed vs Self-Hosted

| Choice | Best when |
|---|---|
| AWS Bedrock | Managed models, governance, fast delivery |
| Bedrock Agents / AgentCore | Managed agent workflows/tools/memory/guardrails |
| SageMaker | Managed training/deployment/model registry |
| EKS + KServe | Kubernetes-native model platform |
| ECS | Simpler container serving |
| Lambda | Lightweight async workflows |
| Ray on AWS | Distributed training/orchestration |
| vLLM/TGI on EC2/EKS | Self-hosted open-weight serving |
| OpenSearch | Hybrid/vector search on AWS |
| S3 | Dataset/model/log storage |
| CloudWatch | Infra/app logs and metrics |
| IAM/KMS/Secrets Manager | Security and access control |

### Bedrock vs Self-Hosted

```text
Bedrock = managed foundation model consumption.
Self-hosted = platform control over open-weight models.
```

**Choose Bedrock when:** governance matters, fast delivery matters, managed scaling matters, custom model internals are not required.

**Choose self-hosted when:** need custom LoRA/QLoRA adapters, need open-weight model, need custom decoding/serving, need unit economics at scale, need full runtime control.

> 📌 **Runnable example:** [`examples/05_bedrock/bedrock_basic.py`](examples/05_bedrock/bedrock_basic.py) · [`examples/04_serving/kserve_manifest.yaml`](examples/04_serving/kserve_manifest.yaml)


## 10.1 — AWS GPU Choices

| Instance family | GPUs | Best for |
|---|---|---|
| g5 | NVIDIA A10G | 7B/13B LoRA, inference, small labs |
| g6 | NVIDIA L4 | Efficient inference, smaller training |
| p4d / p4de | A100 | Large training, distributed finetuning |
| p5 | H100 | High-end training, RLHF, large VLM |
| trn / inf | Trainium / Inferentia | AWS-native managed training/inference |

## 10.2 — Reference AWS GenAI Architecture

```text
Frontend
→ API Gateway / ALB
→ FastAPI / service layer
→ Auth / IAM / Cognito
→ Orchestrator
   ├── Bedrock / self-hosted vLLM
   ├── RAG retriever
   ├── tools/APIs
   ├── memory store
   └── guardrail layer
→ Observability
→ Logs / feedback
→ Offline training/evaluation pipeline
```


## 10.3 — AWS RAG Architecture

```text
S3 document bucket
→ Textract / parser / custom OCR
→ chunking job
→ embedding job
→ OpenSearch / vector DB
→ retriever API
→ reranker
→ LLM generation
→ answer + citations
```

## 10.4 — AWS Fine-Tuning Architecture

```text
S3 dataset
→ SageMaker / EC2 / EKS training job
→ MLflow / W&B / CloudWatch tracking
→ model artifact in S3
→ model registry
→ SageMaker / EKS / vLLM / TGI deployment
→ monitor
```


## 10.5 — Decision Tree & Production Rule

### Which AWS Path?

```text
Quickest path to a working GenAI app?
    → Bedrock + managed RAG + Bedrock Agents

Need fine-tuning on your own data?
    → SageMaker training jobs (managed) or EC2/EKS (custom)

Need Kubernetes-native serving?
    → EKS + KServe + vLLM/TGI

Need distributed training at scale?
    → SageMaker distributed training or Ray on EKS

Need to optimize cost at high QPS?
    → Self-hosted vLLM on EC2 with autoscaling

Need managed vector search?
    → OpenSearch Service / Aurora pgvector / Bedrock Knowledge Base
```

### Production Rule

```text
Bedrock is managed AI consumption.
EKS/KServe/vLLM is AI platform engineering.
Ray/SageMaker/DeepSpeed is training orchestration.
```

For most teams, the realistic path is:

1. **Prototype on Bedrock** (fastest delivery)
2. **Move to self-hosted vLLM** when cost or customization matters
3. **Adopt Bedrock Agents / AgentCore** if you want managed agent infra
4. **Use SageMaker** for serious training jobs
5. **Add OpenSearch / pgvector** for RAG

---

## 📚 Section 10 Summary

```text
Quickest:    Bedrock + Bedrock Agents + Bedrock Knowledge Base (RAG)
Custom FT:   SageMaker / EC2 / EKS
K8s native:  EKS + KServe + vLLM/TGI
Distributed: Ray on AWS / SageMaker distributed / DeepSpeed
Vector:      OpenSearch / Aurora pgvector / Bedrock Knowledge Base
GPU:         g5/g6 (small) → p4 (large FT) → p5 (RLHF / VLM)
```


# Section 11 — Security & Guardrails

> **What this section is for:** Defend against prompt injection, RAG poisoning, memory poisoning, and feedback poisoning with layered guardrails and a safe training data pipeline.


## 11.0 — Main Risks

LLM apps have **attack surfaces traditional apps do not**:

| Risk | Where it happens |
|---|---|
| **Prompt injection** | User prompt / retrieved document / tool output |
| **Data leakage** | Prompt, memory, logs, retrieved chunks |
| **Tool misuse** | Agent calls dangerous API |
| **RAG poisoning** | Malicious document enters vector index |
| **Feedback poisoning** | Bad feedback enters training data |
| **Model poisoning** | Bad curated data updates model weights |
| **Memory poisoning** | Malicious facts stored as memory |
| **Eval poisoning** | Contaminated eval set hides failure |


## 11.1 — Safe Feedback-to-Training Flow

### Unsafe

```text
user feedback → model immediately updates
```

### Safe

```text
user feedback
→ raw logs
→ filtering
→ review
→ curated dataset
→ offline training
→ evaluation
→ model registry
→ canary deploy
→ rollback if needed
```


## 11.2 — Data Storage Layers

```text
1. Raw interaction logs
2. Feedback / preference data
3. Sanitized dataset
4. Curated training dataset
5. Model checkpoint / adapter version
```

Each layer should have **different access controls** and **different retention policies**.


## 11.3 — Preference Data Example

```json
{
  "prompt": "Explain LoRA",
  "chosen": "Correct detailed answer",
  "rejected": "Incorrect or unsafe answer",
  "source": "expert_review",
  "trust_score": 0.95
}
```

The `trust_score` is critical — it lets the training pipeline weight examples by their source quality.


## 11.4 — Poisoning Mitigation Checklist

```text
do not train from raw logs directly
PII redaction
toxicity and jailbreak filtering
prompt injection scanning
trust scoring
human expert review
deduplication
outlier detection
dataset versioning
eval gates
canary deploy
rollback
```


## 11.5 — Guardrail Layers

| Layer | Guardrail |
|---|---|
| **Input** | Prompt injection / PII / abuse detection |
| **Retrieval** | Source trust + metadata permission |
| **Generation** | Policy prompt + output validation |
| **Tool** | Schema validation + permissions + approval |
| **Memory** | Write filters + user scope |
| **Training** | Curated dataset + eval gate |
| **Deployment** | Canary + rollback |


## 11.6 — Decision Tree & Production Rule

### Where Do I Add Guardrails?

```text
User input could be hostile?
    → Input filter (prompt injection + jailbreak detection)

Retrieval returns documents from external sources?
    → Source trust scoring + injection scanning on ingest

Output could leak PII or violate policy?
    → Output validator + PII redaction

Agent calls external APIs?
    → Tool allow-list + permission check + audit log

User data stored long-term?
    → Memory write filter + user scope + expiry

Training on production feedback?
    → Trust score + human review + canary + rollback

Auditors need to know what changed?
    → Dataset versioning + model registry + audit log
```

### Production Rule

```text
Defense in depth:
  - filter at INPUT (cheap, catches most attacks)
  - filter at RETRIEVAL (medium cost, catches RAG poisoning)
  - filter at OUTPUT (expensive, catches leakage)
  - filter at TRAINING (most expensive, prevents backdoors)
  - filter at DEPLOYMENT (canary + rollback catches regressions)

No single layer is enough. Stack them.
```

### Core Rule

```text
Raw feedback is signal.
Curated feedback is training data.
Only reviewed, versioned, evaluated data should touch model weights.
```

---

## 📚 Section 11 Summary

```text
Attack surfaces: prompt injection, data leakage, tool misuse, RAG/feedback/memory/eval poisoning
Defense layers:  input + retrieval + generation + tool + memory + training + deployment
Data pipeline:   raw logs → filtered → reviewed → curated → trained → canary → rollback
Rule:            raw feedback is signal; curated feedback is training data
```


# Section 12 — End-to-End Production Blueprint

> **What this section is for:** The integrated view — how all 11 previous sections fit together into a production system, and the order to build them.


## 12.0 — Complete System Map

```text
User / App
→ API Layer
→ Auth / Tenant / Rate Limit
→ Orchestrator
   ├── Prompt Registry
   ├── RAG Retriever
   ├── Reranker
   ├── Memory
   ├── Tool Calling
   ├── Guardrails
   └── Model Router
→ Model Serving
   ├── Bedrock
   ├── vLLM / TGI
   ├── KServe / SageMaker
   └── fallback model
→ Observability
→ Feedback Collection
→ Offline Evaluation
→ Fine-Tuning / Preference Training
→ Model Registry
→ Deployment / Rollback
```


## 12.1 — End-to-End AI Engineer Roadmap

```text
Step 1:  Understand model architecture          (Section 1)
Step 2:  Choose model for task
Step 3:  Build RAG if knowledge needed         (Section 7)
Step 4:  Add reranker / hybrid / metadata filtering
Step 5:  Add agent workflow only if tools/actions needed   (Section 8)
Step 6:  Add memory only if persistence needed
Step 7:  Add evals before production           (Section 9)
Step 8:  Add observability
Step 9:  Add guardrails/security               (Section 11)
Step 10: Add fine-tuning only when prompting/RAG is not enough   (Section 5)
Step 11: Add preference optimization when behavior quality needs alignment     (Section 6)
Step 12: Serve with managed or self-hosted runtime          (Sections 2 + 10)
Step 13: Monitor, rollback, improve
```


## 12.2 — Decision Tree

```text
Need private documents?
    → RAG                                          (Section 7)

Need domain-specific response format?
    → SFT / LoRA                                  (Section 5)

Need preferred behavior/style?
    → DPO / ORPO / SimPO                          (Section 6)

Need reasoning with verifiable reward?
    → GRPO / verifier-based RL                    (Section 6)

Need visual understanding?
    → VLM / multimodal RAG / VLM fine-tuning      (Section 3)

Need actions/tools?
    → Agent / workflow                            (Section 8)

Need enterprise production?
    → LLMOps + guardrails + observability + AWS   (Sections 9 + 10 + 11)

Need scale/cost control?
    → quantization + serving optimization + caching + batching   (Section 2)
```


## 12.3 — When to Add Each Layer

| Symptom | Add |
|---|---|
| Model hallucinates facts | RAG (Section 7) |
| Wrong style / format | SFT (Section 5) |
| Right info, wrong tone | DPO / ORPO (Section 6) |
| Math / code reasoning failures | GRPO / verifier (Section 6) |
| Can't read documents / charts | VLM / ColPali (Section 3) |
| User wants to do actions | Agent + tools (Section 8) |
| User wants continuity | Memory (Section 8) |
| Hallucinations in production | Eval + observability (Section 9) |
| Security incidents | Guardrails + injection scanning (Section 11) |
| Cost blowing up | Quantization + caching + serving optimization (Section 2) |
| Latency too high | Speculative decoding + prefix caching (Section 2) |


## 12.4 — The Build Order (Pragmatic)

For most enterprise projects, this is the **order of attack**:

```text
PHASE 1 — Foundations (Weeks 1–2)
  - Pick a model (Bedrock for fastest, vLLM+Qwen for open-weight)
  - Pick an evaluation harness
  - Pick a tracing/logging system

PHASE 2 — RAG (Weeks 3–4)
  - Pick chunking strategy
  - Add hybrid search (BM25 + dense)
  - Add a reranker
  - Add metadata filtering
  - Eval retrieval quality

PHASE 3 — Application (Weeks 5–6)
  - Build prompt templates + prompt registry
  - Add input/output guardrails
  - Add observability + tracing
  - Add cost/latency tracking

PHASE 4 — Agents (if needed) (Weeks 7–8)
  - Build tool allow-list + permissions
  - Build state machine (LangGraph-style)
  - Add memory (if needed)
  - Add audit log

PHASE 5 — Adaptation (if needed) (Weeks 9–12)
  - Collect preference data
  - Run DPO/ORPO alignment
  - (Optional) SFT on domain data
  - Re-eval + canary + rollback plan

PHASE 6 — Scale (Weeks 13+)
  - Quantization
  - Speculative decoding
  - Disaggregated serving
  - Multi-region
  - Cost optimization
```


## 12.5 — Final Mental Model

```text
LLM app = model + data + retrieval + tools + memory + evals + serving + security + operations.
```

**Not:** "LLM app = a model + a prompt"

---

## 🎓 Final Words

If you've read every section of this notebook top-to-bottom, you now have a working mental model of:

```text
1.  How modern LLMs are built            (Section 1)
2.  How to serve them fast and cheap    (Section 2)
3.  How to extend them to see/hear      (Section 3)
4.  How to specialize them for code     (Section 4)
5.  How to adapt them to your domain    (Section 5)
6.  How to align them with preferences  (Section 6)
7.  How to connect them to private data (Section 7)
8.  How to give them tools and memory   (Section 8)
9.  How to evaluate and monitor them    (Section 9)
10. How to deploy them on AWS           (Section 10)
11. How to secure them in production    (Section 11)
12. How it all fits together            (Section 12)
```

**Go build something.** 🚀


In [ ]:
# Master notebook metadata
NOTEBOOK_NAME = "End-to-End LLM/VLM/RAG/AgentOps/LLMOps Master Notebook"
VERSION = "V1.4 Master"
LAST_UPDATED = "2026-06-22"

SECTIONS = [
    "1. LLM Architecture",
    "2. Attention & Serving Kernels",
    "3. VLM & Multimodal",
    "4. Coding Models",
    "5. Fine-Tuning & PEFT",
    "6. Alignment & RLHF",
    "7. RAGOps",
    "8. AgentOps",
    "9. LLMOps & EvalOps",
    "10. AWS Production",
    "11. Security & Guardrails",
    "12. End-to-End Blueprint",
]

print(f"{NOTEBOOK_NAME}")
print(f"Version: {VERSION}")
print(f"Last updated: {LAST_UPDATED}")
print(f"\n{len(SECTIONS)} sections:")
for s in SECTIONS:
    print(f"  - {s}")
